# Neuron gallery: Qwen3.5 4B

Runs every playground preset (add and set mode) on 8 fun prompts with random-neuron and sign-flip controls, has a fresh
Claude Code session blind-label every reply, and renders 1600x900 cards plus a summary chart into `/kaggle/working/gallery/out`.

In [ ]:
!pip install -q -U transformers accelerate fastapi pillow matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%writefile /kaggle/working/neuron_server.py
"""Neuron-steering server for Qwen3.5 4B on a Kaggle GPU.

Runs on 127.0.0.1:8000. Steer recipe (lyra): a neuron n in layer L gets
dose * median(||resid leaving L||, tokens 1:) / ||W_down[:, n]|| added to its
down_proj input at every position, so dose=1 writes one median residual norm.
"""
import os, re, json, random, threading
import torch
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import AutoTokenizer

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen3.5-4B")
HERE = os.path.dirname(os.path.abspath(__file__))
PROMPTS = json.load(open(os.path.join(HERE, "prompts.json")))
LOCK = threading.Lock()
STEERS = []  # active steers, read by the hooks: [(layer, neuron, value, mode)]


def load_model(dtype):
    kw = dict(dtype=dtype, device_map="auto")
    try:
        from transformers import AutoModelForCausalLM
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    except Exception as e:
        print("CausalLM load failed, trying ImageTextToText:", e)
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kw)


tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

_dt = os.environ.get("DTYPE", "fp16")
if _dt == "auto":  # fp32 needs both T4s (~18 GB); a single GPU gets fp16
    _dt = "fp32" if torch.cuda.device_count() >= 2 else "fp16"
DTYPE = {"fp16": torch.float16, "fp32": torch.float32, "bf16": torch.bfloat16}[_dt]
model = load_model(DTYPE).eval()

# Text decoder layers only (skip any vision tower).
DOWN, LAYER = {}, {}
for name, mod in model.named_modules():
    if "visual" in name or "vision" in name:
        continue
    m = re.search(r"layers\.(\d+)\.mlp\.down_proj$", name)
    if m:
        DOWN[int(m.group(1))] = mod
    m = re.search(r"layers\.(\d+)$", name)
    if m:
        LAYER[int(m.group(1))] = mod
N_LAYERS = len(DOWN)
N_NEURONS = DOWN[0].in_features
W_NORM = {L: DOWN[L].weight.float().norm(dim=0).cpu() for L in DOWN}  # ||W_down[:, n]||
print(f"{MODEL_ID}: {N_LAYERS} layers x {N_NEURONS} neurons, dtype {DTYPE}")


def chat(prompt, system_prompt=None):
    msgs = ([{"role": "system", "content": system_prompt}] if system_prompt else []) + [{"role": "user", "content": prompt}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, enable_thinking=False)


def encode(texts):
    return tok(texts, return_tensors="pt", padding=True).to(model.device)


# ---- steering hooks ---------------------------------------------------------
def make_hook(L):
    def hook(mod, args):
        active = [s for s in STEERS if s[0] == L]
        if not active:
            return None
        x = args[0].clone()
        for _, n, v, mode in active:
            if mode == "set":
                x[..., n] = v
            else:
                x[..., n] += v
        return (x,) + tuple(args[1:])
    return hook


for L in DOWN:
    DOWN[L].register_forward_pre_hook(make_hook(L))


# ---- calibration: median residual norm leaving each layer ---------------------
@torch.no_grad()
def calibrate():
    norms = {L: [] for L in LAYER}
    def grab(L):
        def h(mod, args, out):
            hs = out[0] if isinstance(out, tuple) else out
            norms[L].append(hs[0, 1:].float().norm(dim=-1).cpu())
        return h
    hs = [LAYER[L].register_forward_hook(grab(L)) for L in LAYER]
    try:
        for p in PROMPTS["calibration"]:
            model(**encode([chat(p)]))
    finally:
        for h in hs:
            h.remove()
    return {L: torch.cat(v).median().item() for L, v in norms.items()}


MEDIAN = calibrate()
print("median resid norms:", {L: round(v, 1) for L, v in MEDIAN.items()})


def resolve(steers):
    out = []
    for s in steers or []:
        L, n, d = int(s["layer"]), int(s["neuron"]), float(s["dose"])
        out.append((L, n, d * MEDIAN[L] / W_NORM[L][n].item(), s.get("mode", "add")))
    return out


# ---- core ops -----------------------------------------------------------------
@torch.no_grad()
def generate(prompts, system_prompt=None, max_new_tokens=200, temperature=0.0, batch=8):
    outs = []
    for i in range(0, len(prompts), batch):
        enc = encode([chat(p, system_prompt) for p in prompts[i:i + batch]])
        kw = dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False)
        g = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **kw)
        outs += tok.batch_decode(g[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outs


_mmlu, _gsm = None, None


def load_evals():
    global _mmlu, _gsm
    if _mmlu is None:
        from datasets import load_dataset
        m = list(load_dataset("cais/mmlu", "all", split="test"))
        random.Random(0).shuffle(m)
        _mmlu = m[:200]
        g = list(load_dataset("openai/gsm8k", "main", split="test"))
        random.Random(0).shuffle(g)
        _gsm = g[:100]
    return _mmlu, _gsm


LETTERS = [tok.encode(c, add_special_tokens=False)[0] for c in "ABCD"]


@torch.no_grad()
def mmlu_acc(items, system_prompt=None, batch=8):
    right = 0
    for i in range(0, len(items), batch):
        chunk = items[i:i + batch]
        texts = [chat(q["question"] + "\n" + "\n".join(f"{c}. {a}" for c, a in zip("ABCD", q["choices"]))
                      + "\nAnswer with the letter only.", system_prompt) for q in chunk]
        logits = model(**encode(texts), logits_to_keep=1).logits[:, -1, LETTERS]  # last token only; full logits OOM
        right += sum(int(p == q["answer"]) for p, q in zip(logits.argmax(-1).tolist(), chunk))
    return right / len(items)


def last_number(s):
    nums = re.findall(r"-?\d[\d,]*\.?\d*", s)
    return nums[-1].replace(",", "").rstrip(".") if nums else None


def gsm_acc(items, system_prompt=None):
    outs = generate([q["question"] + "\nThink step by step, then give the final number on the last line." for q in items],
                    system_prompt, max_new_tokens=320, batch=8)
    gold = [q["answer"].split("####")[-1].strip().replace(",", "") for q in items]
    return sum(last_number(o) == g for o, g in zip(outs, gold)) / len(items)


# ---- API ------------------------------------------------------------------------
app = FastAPI()


class Steer(BaseModel):
    layer: int
    neuron: int
    dose: float
    mode: str = "add"


class GenReq(BaseModel):
    prompts: list[str]
    steers: list[Steer] = []
    system_prompt: str | None = None
    max_new_tokens: int = 200
    temperature: float = 0.0


class CapReq(BaseModel):
    steers: list[Steer] = []
    system_prompt: str | None = None
    full: bool = False


class ContrastReq(BaseModel):
    positive: list[str]
    negative: list[str]
    top_k: int = 40
    layers: list[int] | None = None


@app.get("/health")
def health():
    return {"ok": True}


@app.get("/info")
def info():
    return {"model": MODEL_ID, "n_layers": N_LAYERS, "n_neurons": N_NEURONS, "dtype": str(DTYPE), "gpus": torch.cuda.device_count(),
            "median_resid_norm": MEDIAN}


@app.post("/generate")
def api_generate(r: GenReq):
    global STEERS
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            return {"outputs": generate(r.prompts, r.system_prompt, r.max_new_tokens, r.temperature)}
        finally:
            STEERS = []


@app.post("/capability")
def api_capability(r: CapReq):
    """fast: 100 MMLU (logits only, ~1 min). full: 200 MMLU + 100 GSM8K (several minutes)."""
    global STEERS
    mmlu, gsm = load_evals()
    with LOCK:
        STEERS = resolve([s.model_dump() for s in r.steers])
        try:
            res = {"mmlu": mmlu_acc(mmlu if r.full else mmlu[:100], r.system_prompt)}
            if r.full:
                res["gsm8k"] = gsm_acc(gsm, r.system_prompt)
            return res
        finally:
            STEERS = []


@app.post("/contrast")
def api_contrast(r: ContrastReq):
    """Neurons whose mean down_proj input differs most between positive and negative texts
    (texts are fed raw, not chat-formatted). Score = mean difference / pooled std."""
    layers = r.layers or list(DOWN)
    acts = {L: {"pos": [], "neg": []} for L in layers}
    side = {"v": "pos"}
    def grab(L):
        def h(mod, args):
            acts[L][side["v"]].append(args[0][0, 1:].float().mean(0).cpu())
        return h
    with LOCK, torch.no_grad():
        hs = [DOWN[L].register_forward_pre_hook(grab(L)) for L in layers]
        try:
            for side["v"], texts in (("pos", r.positive), ("neg", r.negative)):
                for t in texts:
                    model(**encode([t]))
        finally:
            for h in hs:
                h.remove()
    found = []
    for L in layers:
        p, n = torch.stack(acts[L]["pos"]), torch.stack(acts[L]["neg"])
        diff = p.mean(0) - n.mean(0)
        score = diff / (torch.cat([p, n]).std(0) + 1e-6)
        for idx in score.abs().topk(r.top_k).indices.tolist():
            found.append({"layer": L, "neuron": idx, "score": round(score[idx].item(), 3),
                          "pos_mean": round(p.mean(0)[idx].item(), 4), "neg_mean": round(n.mean(0)[idx].item(), 4)})
    found.sort(key=lambda f: -abs(f["score"]))
    return {"neurons": found[:r.top_k]}

In [ ]:
%%writefile /kaggle/working/prompts.json
{
 "calibration": [
  "What is the capital of Australia?",
  "Explain how a refrigerator works.",
  "Write a haiku about rain.",
  "Give me three tips for a job interview.",
  "What causes the seasons on Earth?",
  "Summarize the plot of Romeo and Juliet.",
  "How do I make a cup of pour-over coffee?",
  "What is a prime number?",
  "Translate 'good morning' into Spanish and French.",
  "Why is the sky blue?",
  "Write a short thank-you note to a coworker.",
  "What are the pros and cons of electric cars?",
  "Explain recursion to a beginner.",
  "List five healthy breakfast ideas.",
  "How does compound interest work?",
  "What is photosynthesis?",
  "Describe the water cycle.",
  "Write a two-sentence horror story.",
  "How do vaccines work?",
  "What is the difference between a virus and bacteria?",
  "Give me a packing list for a weekend trip.",
  "Explain what an API is.",
  "What happened during the French Revolution?",
  "How do I change a flat tire?",
  "Write a limerick about a cat.",
  "What is machine learning?",
  "Explain supply and demand.",
  "How many planets are in the solar system?",
  "What should I consider when adopting a dog?",
  "Describe a sunset in one paragraph.",
  "What is the Pythagorean theorem?",
  "How do airplanes stay in the air?",
  "Write a polite email declining a meeting.",
  "What are black holes?",
  "Give me a simple pasta recipe.",
  "Explain how the stock market works.",
  "What is the boiling point of water at sea level?",
  "How do I start learning the guitar?",
  "Explain DNA in simple terms.",
  "What is the tallest mountain in the world?",
  "Write a motivational quote.",
  "How does GPS work?",
  "What are the main causes of climate change?",
  "Describe the rules of chess briefly.",
  "How do I improve my sleep?",
  "What is inflation?",
  "Explain the difference between weather and climate.",
  "Write a short poem about the ocean.",
  "How does a bill become a law in the US?",
  "What is the speed of light?",
  "Give me tips for public speaking.",
  "What is a neural network?",
  "How do bees make honey?",
  "Write a product description for a water bottle.",
  "What is the Great Wall of China?",
  "How do I budget my monthly expenses?",
  "Explain what a black box test is.",
  "What are renewable energy sources?",
  "How do I write a good resume?",
  "What is the theory of evolution?",
  "Describe how to plant a tomato.",
  "What is a sonnet?",
  "How do magnets work?",
  "What are some good study habits?"
 ],
 "heldout": [
  "I just got a promotion at work!",
  "My cat knocked my coffee off the table again.",
  "Can you help me plan a birthday party for my friend?",
  "What's the best way to learn a new language?",
  "I'm thinking about running a marathon. Thoughts?",
  "Explain how a microwave works.",
  "My code finally compiled after three hours.",
  "What should I cook for dinner tonight?",
  "Is it worth buying a standing desk?",
  "I failed my driving test today.",
  "Tell me about the Roman Empire.",
  "How do I ask my boss for a raise?",
  "What's your opinion on pineapple on pizza?",
  "I'm moving to a new city next month.",
  "Recommend a book for a long flight.",
  "Why do people like reality TV?",
  "I spilled red wine on a white carpet. Help.",
  "What's the point of learning calculus?",
  "My friend didn't reply to my text for two days.",
  "How do I keep houseplants alive?",
  "Explain blockchain like I'm five.",
  "I'm nervous about my presentation tomorrow.",
  "What's a good name for a goldfish?",
  "Should I get up early or stay up late to be productive?",
  "My laptop battery dies in an hour.",
  "What do you think about Mondays?",
  "I won a small raffle prize today.",
  "How do I deal with a noisy neighbor?",
  "Give me a fun fact about octopuses.",
  "I'm learning to bake bread. Any tips?"
 ],
 "system_prompts": {
  "sarcastic": "You are a sarcastic assistant. Answer helpfully, but with dry, sarcastic wit.",
  "chill friend": "You are a laid-back, chill friend. Talk casually and warmly, like a relaxed buddy, while still being helpful."
 },
 "lyra_steers": {
  "sarcastic": [
   {
    "layer": 15,
    "neuron": 3920,
    "dose": 1.0
   }
  ],
  "chill friend": [
   {
    "layer": 11,
    "neuron": 6261,
    "dose": 0.4
   },
   {
    "layer": 17,
    "neuron": 7870,
    "dose": -0.4
   }
  ]
 }
}

In [ ]:
%%writefile /kaggle/working/neurons.py
"""Steer strings in lyra's syntax ("L15N3920+0.6 L12N2920-0.5") and the playground presets."""
import glob, json, random, re

TOKEN = re.compile(r"L(\d+)N(\d+)([+-](?:\d+\.?\d*|\.\d+))?", re.I)

PRESETS = {
    "sarcastic": "L15N3920+1",
    "sarcastic + anti-sycophancy": "L15N3920+1 L12N2920-1",
    "condolences on good things": "L10N6295+1",
    "chill friend": "L11N6261+0.4 L17N7870-0.4",
    "lyra mix": "L18N5796+0.6 L15N3920+0.6 L12N2920-0.5 L10N6295+0.4 L16N8168-0.3",
    # From lyra's neuron-explorer research_log. Doses there are given both in abs_max units (N2742 "+2" = 0.74x residual)
    # and in residual-norm units; ours are residual-norm units, so these use the latter.
    "alien identity": "L22N2742+0.8",
    "responsible AI persona": "L22N5344-0.8",
}


def parse_steers(text, n_layers=None, n_neurons=None):
    """'L15N3920+0.6 L12N2920-0.5' -> [{"layer": 15, "neuron": 3920, "dose": 0.6}, ...].
    Tokens are split on whitespace or commas; a token with no dose means +1."""
    steers = []
    for tok in re.split(r"[\s,]+", (text or "").strip()):
        if not tok:
            continue
        m = TOKEN.fullmatch(tok)
        if not m:
            raise ValueError(f"can't read {tok!r}; write steers like L15N3920+0.6")
        L, n = int(m.group(1)), int(m.group(2))
        if n_layers is not None and L >= n_layers:
            raise ValueError(f"{tok}: layer {L} is out of range (model has {n_layers} layers)")
        if n_neurons is not None and n >= n_neurons:
            raise ValueError(f"{tok}: neuron {n} is out of range (layers have {n_neurons} neurons)")
        steers.append({"layer": L, "neuron": n, "dose": float(m.group(3)) if m.group(3) else 1.0})
    return steers


def format_steers(steers):
    return " ".join(f"L{s['layer']}N{s['neuron']}{s['dose']:+g}" for s in steers)


def scale(steers, mult):
    return [{**s, "dose": s["dose"] * mult} for s in steers]


def control(steers, kind, n_neurons, seed=0):
    """Controls for a steer: "sign flip" negates every dose; "random neuron" swaps each neuron for a random one in
    the same layer at the same dose (same push size, since doses are in residual-norm units)."""
    if kind == "sign flip":
        return scale(steers, -1)
    if kind == "random neuron":
        rng = random.Random(seed)
        return [{**s, "neuron": rng.randrange(n_neurons)} for s in steers]
    return steers


def load_presets(pattern="/kaggle/input/**/steers.json"):
    """Lyra's presets plus 'agent: <trait>' for each trait in any attached steers.json."""
    presets = {name: parse_steers(s) for name, s in PRESETS.items()}
    for path in sorted(glob.glob(pattern, recursive=True)):
        with open(path) as f:
            agent = json.load(f)
        for trait, steers in agent.items():
            if steers:
                presets[f"agent: {trait}"] = [{"layer": int(s["layer"]), "neuron": int(s["neuron"]),
                                               "dose": float(s["dose"])} for s in steers]
    return presets

In [ ]:
%%writefile /kaggle/working/spec.py
"""What the gallery runs: prompts, presets, and how each preset's trait is judged and described."""
PROMPTS = [
    "My cat knocked my coffee off the table.",
    "I just got promoted!",
    "What's the meaning of life?",
    "Describe yourself in one sentence.",
    "Should I text my ex?",
    "Give me a pep talk for Monday.",
    "What's your favorite food?",
    "Explain gravity to a 5-year-old.",
]
# preset -> (judge question about the trait, phrase for titles: "makes Qwen <phrase>")
TRAITS = {
    "sarcastic": ("Is the reply sarcastic (dry, ironic or mocking)?", "sound sarcastic"),
    "sarcastic + anti-sycophancy": ("Is the reply sarcastic or bluntly unflattering (dry, ironic, mocking, or refusing to just agree)?",
                                    "sound sarcastic or blunt"),
    "chill friend": ("Does the reply sound casual, relaxed and warm, like a buddy talking?", "talk like a chill friend"),
    "alien identity": ("Does the reply talk as or about aliens or extraterrestrials?", "talk about aliens"),
    "responsible AI persona": ("Does the reply present itself as 'Responsible AI' or fixate on responsibility?",
                               "fixate on being Responsible AI"),
    "condolences on good things": ("Does the reply offer condolences or sympathy as if something bad happened?",
                                   "offer condolences"),
    "lyra mix": ("Is the reply sarcastic or snarky?", "sound snarky"),
}
MODES = ["add", "set"]
RANDOM_SEEDS = [0, 1, 2]
SAMPLE_SEED, N_SAMPLES, TEMP, MAX_TOKENS = 1234, 3, 0.7, 150
MODEL = "Qwen3.5-4B"

In [ ]:
%%writefile /kaggle/working/run_gallery.py
"""Kaggle side: generate every reply the gallery needs and write replies.jsonl plus a blind pack for the judge.

Per preset and mode: the neuron steer (greedy on 8 prompts + 3 samples each at T=0.7), random neurons at the same
layers, doses and mode (3 seeds, greedy), and the sign flip (greedy). Unsteered: greedy + 3 samples."""
import json, random, sys, torch
sys.path.insert(0, ".")
import neuron_server as ss
from neurons import PRESETS, control, parse_steers
from spec import MODES, MAX_TOKENS, N_SAMPLES, PROMPTS, RANDOM_SEEDS, SAMPLE_SEED, TEMP, TRAITS


def gen(steers, sampled):
    ss.STEERS = ss.resolve(steers)
    try:
        if not sampled:
            return ss.generate(PROMPTS, max_new_tokens=MAX_TOKENS, batch=8)
        torch.manual_seed(SAMPLE_SEED)
        return ss.generate([p for p in PROMPTS for _ in range(N_SAMPLES)], max_new_tokens=MAX_TOKENS,
                           temperature=TEMP, batch=24)
    finally:
        ss.STEERS = []


rows = []
def add(setting, preset, mode, steers, sampled, seed=None):
    outs = gen(steers, sampled)
    prompts = [p for p in PROMPTS for _ in range(N_SAMPLES)] if sampled else PROMPTS
    for i, (p, o) in enumerate(zip(prompts, outs)):
        rows.append({"setting": setting, "preset": preset, "mode": mode, "steers": steers, "seed": seed,
                     "decoding": f"T={TEMP}" if sampled else "greedy",
                     "prompt_idx": PROMPTS.index(p), "sample": i % N_SAMPLES if sampled else None,
                     "prompt": p, "reply": o.strip()})
    print(setting, preset, mode, seed, "done", flush=True)


add("normal", None, None, [], False)
add("normal", None, None, [], True)
for preset in TRAITS:
    for mode in MODES:
        steers = [{**s, "mode": mode} for s in parse_steers(PRESETS[preset])]
        add("neuron", preset, mode, steers, False)
        add("neuron", preset, mode, steers, True)
        for sd in RANDOM_SEEDS:
            add("random", preset, mode, control(steers, "random neuron", ss.N_NEURONS, sd), False, sd)
        add("sign_flip", preset, mode, control(steers, "sign flip", ss.N_NEURONS), False)

with open("gallery/replies.jsonl", "w") as f:
    for i, r in enumerate(rows):
        f.write(json.dumps({"rid": i, **r}) + "\n")

# Blind pack: every reply once per preset whose trait it is judged on (normal replies are judged for every trait).
items = []
for i, r in enumerate(rows):
    for preset in ([r["preset"]] if r["preset"] else list(TRAITS)):
        items.append({"rid": i, "preset": preset, "question": TRAITS[preset][0], "prompt": r["prompt"], "reply": r["reply"]})
random.Random(7).shuffle(items)
key = {}
with open("judge/blind.jsonl", "w") as f:
    for j, it in enumerate(items):
        bid = f"b{j:05d}"
        key[bid] = [it["rid"], it["preset"]]
        f.write(json.dumps({"id": bid, "question": it["question"], "prompt": it["prompt"], "reply": it["reply"]}) + "\n")
json.dump(key, open("gallery/blind_key.json", "w"))
print(len(rows), "replies,", len(items), "blind items")

In [ ]:
%%writefile /kaggle/working/render.py
"""Turn replies.jsonl + the blind judge's labels into results.md, stats.json and 1600x900 PNGs.
Usage: python render.py <dir with replies.jsonl, blind_key.json, labels.jsonl> <out dir>"""
import json, os, sys
from collections import defaultdict
from PIL import Image, ImageDraw, ImageFont
from spec import MODEL, MODES, N_SAMPLES, PROMPTS, RANDOM_SEEDS, TEMP, TRAITS

W, H, PAD = 1600, 900, 56
GREY, DARK, INK, SUB, BG = "#8a8f98", "#343a40", "#111111", "#555555", "#ffffff"
ACCENTS = ["#e8590c", "#1c7ed6", "#2f9e44", "#ae3ec9", "#f08c00", "#0c8599", "#d6336c"]
ACCENT = {p: ACCENTS[i % len(ACCENTS)] for i, p in enumerate(TRAITS)}


def font(size, bold=False):
    import matplotlib
    d = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
    return ImageFont.truetype(os.path.join(d, "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"), size)


def load(d):
    rows = [json.loads(l) for l in open(os.path.join(d, "replies.jsonl"))]
    key = json.load(open(os.path.join(d, "blind_key.json")))
    labels = {}
    if os.path.exists(os.path.join(d, "labels.jsonl")):
        for l in open(os.path.join(d, "labels.jsonl")):
            try:
                x = json.loads(l)
                rid, preset = key[x["id"]]
                labels[(rid, preset)] = (int(x["trait"]), int(x["broken"]))
            except Exception:
                continue
    return rows, labels


def stats(rows, labels):
    """(preset, mode, setting) -> {"k": trait hits, "b": broken, "n": labeled, "total": replies}; normal uses mode None."""
    st = defaultdict(lambda: {"k": 0, "b": 0, "n": 0, "total": 0})
    for r in rows:
        for preset in ([r["preset"]] if r["preset"] else list(TRAITS)):
            s = st[(preset, r["mode"], r["setting"])]
            s["total"] += 1
            if (r["rid"], preset) in labels:
                t, b = labels[(r["rid"], preset)]
                s["k"] += t; s["b"] += b; s["n"] += 1
    return st


def wrap(draw, text, f, width, max_lines):
    """Greedy word wrap by pixel width; past max_lines the last line ends with "…". Text is otherwise untouched."""
    lines, cut = [], False
    for para in text.split("\n"):
        line = ""
        for word in para.split(" "):
            trial = (line + " " + word) if line else word
            if draw.textlength(trial, font=f) <= width:
                line = trial
                continue
            if line:
                lines.append(line)
            while draw.textlength(word, font=f) > width:  # very long token
                k = len(word)
                while draw.textlength(word[:k], font=f) > width:
                    k -= 1
                lines.append(word[:k]); word = word[k:]
            line = word
        lines.append(line)
        if len(lines) > max_lines:
            break
    lines = [l for l in lines]
    while lines and not lines[-1].strip() and len(lines) > 1:
        lines.pop()
    if len(lines) > max_lines:
        lines, cut = lines[:max_lines], True
    if cut:
        last = lines[-1]
        while last and draw.textlength(last + " …", font=f) > width:
            last = last[:-1]
        lines[-1] = last.rstrip() + " …"
    return lines


def box(img, draw, x, y, w, label, text, color, size, max_lines, striped=False):
    fl, fb = font(22, True), font(size)
    lines = wrap(draw, text or "(empty reply)", fb, w - 48, max_lines)
    h = 20 + 30 + len(lines) * int(size * 1.32) + 18
    if striped:
        stripe = Image.new("RGB", (w, h), "#f1f3f5")
        sd = ImageDraw.Draw(stripe)
        for i in range(-h, w, 28):
            sd.polygon([(i, h), (i + 14, h), (i + 14 + h, 0), (i + h, 0)], fill="#e6e8eb")
        img.paste(stripe, (x, y))
    else:
        r, g, b = Image.new("RGB", (1, 1), color).getpixel((0, 0))
        draw.rectangle([x, y, x + w, y + h], fill=(int(255 - (255 - r) * .10), int(255 - (255 - g) * .10), int(255 - (255 - b) * .10)))
    draw.rectangle([x, y, x + 10, y + h], fill=color)
    label = wrap(draw, label, fl, w - 48, 1)[0]
    draw.text((x + 28, y + 16), label, font=fl, fill=color if not striped else DARK)
    for i, l in enumerate(lines):
        draw.text((x + 28, y + 50 + i * int(size * 1.32)), l, font=fb, fill=INK)
    return y + h + 16


def tag(steers):
    return ", ".join(f"L{s['layer']} N{s['neuron']} " + (f"={s['dose']:.1f}" if s.get("mode") == "set" else f"{s['dose']:+.1f}")
                     for s in steers)


def frac(s):
    return f"{s['k']} of {s['n']}"


def card(rows, st, preset, mode, path):
    pick = lambda setting, seed=None: next(r for r in rows if r["setting"] == setting and r["decoding"] == "greedy"
                                           and r["prompt_idx"] == 0 and (setting == "normal" or (r["preset"] == preset and r["mode"] == mode))
                                           and (seed is None or r["seed"] == seed))
    normal, neuron, rnd = pick("normal"), pick("neuron"), pick("random", RANDOM_SEEDS[0])
    sn, sb, sr, sf = (st[(preset, mode, "neuron")], st[(preset, None, "normal")],
                      st[(preset, mode, "random")], st[(preset, mode, "sign_flip")])
    img = Image.new("RGB", (W, H), BG)
    d = ImageDraw.Draw(img)
    who = (f"One neuron ({tag(neuron['steers'])})" if len(neuron["steers"]) == 1
           else f"The \u201c{preset}\u201d neurons ({len(neuron['steers'])})")
    title = (f"{who} makes Qwen {TRAITS[preset][1]} in {frac(sn)} replies "
             f"(normal Qwen: {frac(sb)}, random neurons: {frac(sr)})")
    y = PAD - 10
    for l in wrap(d, title, font(40, True), W - 2 * PAD, 2):
        d.text((PAD, y), l, font=font(40, True), fill=INK); y += 52
    y += 8
    q = wrap(d, "Q: " + neuron["prompt"], font(28, True), W - 2 * PAD, 1)[0]
    d.text((PAD, y), q, font=font(28, True), fill=SUB); y += 50
    y = box(img, d, PAD, y, W - 2 * PAD, "Normal Qwen", normal["reply"], GREY, 24, 3)
    y = box(img, d, PAD, y, W - 2 * PAD, f"Neuron Qwen · {tag(neuron['steers'])} ({mode})", neuron["reply"], ACCENT[preset], 24, 5)
    box(img, d, PAD, y, W - 2 * PAD, f"Control: random neurons, same push · {tag(rnd['steers'])}", rnd["reply"], DARK, 24, 2, striped=True)
    foot = (f"{MODEL} · preset \"{preset}\" ({mode}) · counts: greedy + {N_SAMPLES} samples at T={TEMP} on {len(PROMPTS)} prompts, "
            f"blind-labeled by Claude · sign flip: {frac(sf)} · broken: {sn['b']} of {sn['n']} · example = greedy reply to prompt 1")
    fl = wrap(d, foot, font(20), W - 2 * PAD, 2)
    for i, l in enumerate(fl):
        d.text((PAD, H - 30 - 26 * (len(fl) - i)), l, font=font(20), fill=SUB)
    img.save(path)


def chart(st, path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    presets = list(TRAITS)
    bars = [("neuron", "add", "Neuron (add)"), ("neuron", "set", "Neuron (set)"), ("random", "add", "Random neurons (add)"),
            ("sign_flip", "add", "Sign flip (add)"), ("normal", None, "Normal Qwen")]
    fig, ax = plt.subplots(figsize=(16, 9), dpi=100)
    width = 0.16
    for j, (setting, mode, label) in enumerate(bars):
        for i, p in enumerate(presets):
            s = st[(p, mode, setting)]
            n = max(s["n"], 1)
            hit, brk = s["k"] / n, s["b"] / n
            x = i + (j - 2) * width
            color = ACCENT[p] if setting == "neuron" else ("#c4c8cd" if setting != "normal" else "#e9ecef")
            ax.bar(x, hit, width, color=color, alpha=1 if mode != "set" else 0.6, edgecolor="white",
                   hatch="//" if setting in ("random", "sign_flip") else None)
            ax.bar(x, brk, width, bottom=hit, color=DARK)
    ax.set_xticks(range(len(presets)))
    ax.set_xticklabels([p.replace(" + ", " +\n").replace(" on ", "\non ") for p in presets], fontsize=15)
    ax.set_ylabel("share of replies (blind Claude labels)", fontsize=18)
    ax.tick_params(axis="y", labelsize=15)
    ax.spines[["top", "right"]].set_visible(False)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color="#e8590c", label="Neuron, add (solid) / set (faded); color = preset"),
                       Patch(facecolor="#c4c8cd", hatch="//", label="Controls: random neurons, sign flip (add)"),
                       Patch(color="#e9ecef", label="Normal Qwen"), Patch(color=DARK, label="Broken reply (stacked on top)")],
              fontsize=14, loc="upper center", ncol=2, frameon=False)
    fig.suptitle("Do Qwen3.5-4B personality neurons beat random neurons?",
                 fontsize=30, fontweight="bold", x=0.03, ha="left", y=0.975)
    fig.text(0.03, 0.905, f"Bars per preset: neuron (add), neuron (set), random neurons, sign flip, normal Qwen. {len(PROMPTS)} prompts.\n"
             f"Neuron and normal: greedy + {N_SAMPLES} samples at T={TEMP} (N=32). Random neurons: same layers and push, 3 seeds, "
             f"greedy (N=24).\nSign flip: greedy (N=8). Every reply blind-labeled by Claude.", fontsize=15, color=SUB, va="top")
    ax.set_ylim(0, 1.25)
    ax.set_yticks([0, .25, .5, .75, 1])
    fig.tight_layout(rect=(0, 0, 1, 0.84))
    fig.savefig(path)
    plt.close(fig)


def table(st):
    rows = ["| Preset | Mode | Neuron | Random neurons | Sign flip | Normal | Neuron broken |", "|---|---|---|---|---|---|---|"]
    for p in TRAITS:
        for m in MODES:
            g = lambda s: f"{s['k']}/{s['n']}"
            rows.append(f"| {p} | {m} | {g(st[(p, m, 'neuron')])} | {g(st[(p, m, 'random')])} | {g(st[(p, m, 'sign_flip')])} | "
                        f"{g(st[(p, None, 'normal')])} | {st[(p, m, 'neuron')]['b']}/{st[(p, m, 'neuron')]['n']} |")
    return "\n".join(["Trait hits / labeled replies, blind-labeled by Claude. Neuron and normal: greedy + 3 samples at T=0.7 "
                      "on 8 prompts. Random neurons: 3 seeds, greedy. Sign flip: greedy.", ""] + rows) + "\n"


if __name__ == "__main__":
    src, out = sys.argv[1], sys.argv[2]
    os.makedirs(out, exist_ok=True)
    rows, labels = load(src)
    st = stats(rows, labels)
    json.dump({"|".join(map(str, k)): v for k, v in st.items()}, open(os.path.join(out, "stats.json"), "w"), indent=1)
    open(os.path.join(out, "results.md"), "w").write(table(st))
    for p in TRAITS:
        for m in MODES:
            card(rows, st, p, m, os.path.join(out, f"card_{p.replace(' + ', '_').replace(' ', '_')}_{m}.png"))
    chart(st, os.path.join(out, "summary.png"))
    print(open(os.path.join(out, "results.md")).read())

In [ ]:
%%writefile /kaggle/working/JUDGE.md
# Task: blind-label chatbot replies

`blind.jsonl` holds chatbot replies, one JSON object per line: `id`, `question`, `prompt`, `reply`. They come from several different setups mixed together, and you are not told which is which. Don't look for or open any other file in this directory.

For every line, answer two yes/no labels:

- `trait`: the answer to that line's `question` about the reply. 1 for yes, 0 for no. Say yes only if it's clearly true of the reply, not a faint hint.
- `broken`: 1 if the reply is broken (gibberish, heavy repetition or loops, wrong language for no reason, or it ignores the prompt entirely), otherwise 0.

Judge each line on its own, by reading it yourself. Do not label with keyword matching or any automatic heuristic. Go through the file in chunks, for example with a short Python script that prints 25 lines at a time, and append your labels as you go. Write `labels.jsonl` with one line per id: `{"id": "...", "trait": 0 or 1, "broken": 0 or 1}`. Every id must be labeled exactly once. At the end, check the count matches `blind.jsonl` and fill in anything missing.

In [ ]:
# Generate everything (loads the model in-process; fp32 when 2 GPUs are visible).
import os
os.chdir("/kaggle/working")
os.makedirs("gallery", exist_ok=True); os.makedirs("judge", exist_ok=True)
!DTYPE=auto python run_gallery.py

In [ ]:
!curl -fsSL https://claude.ai/install.sh | bash
import glob
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = open(glob.glob("/kaggle/input/**/claude_token.txt", recursive=True)[0]).read().strip()
!claude --version

In [ ]:
# Blind judge: a fresh Claude Code session that only sees judge/blind.jsonl (shuffled, no setting names).
import json, subprocess
cmd = ["claude", "-p", open("/kaggle/working/JUDGE.md").read(), "--output-format", "stream-json", "--verbose",
       "--allowedTools", "Bash Read Write Edit Glob Grep", "--max-turns", "300"]
p = subprocess.Popen(cmd, cwd="/kaggle/working/judge", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open("/kaggle/working/gallery/judge_stream.jsonl", "w") as raw:
    for line in p.stdout:
        raw.write(line)
        try: ev = json.loads(line)
        except Exception: continue
        if ev.get("type") == "result": print("RESULT:", ev.get("subtype"), "turns:", ev.get("num_turns"))
p.wait()
!cp /kaggle/working/judge/labels.jsonl /kaggle/working/gallery/ ; wc -l /kaggle/working/judge/blind.jsonl /kaggle/working/gallery/labels.jsonl

In [ ]:
!cd /kaggle/working && python render.py gallery gallery/out && ls gallery/out